In [15]:
# !{sys.executable} -m pip install "accelerate>=1.1.0"
# !pip install -U sentence-transformers
# !pip install -U accelerate
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (classification_report, f1_score, precision_score,
                             recall_score, accuracy_score)
from sklearn.preprocessing import normalize
import warnings
warnings.filterwarnings('ignore')

# sentence-based embedding approaches for Twitter Climate Change Sentiment Dataset
# NOTE: takes about 4 minutes to run

# NOTE: Used ChatGPT/Google Gemini to assist with writing code for this step
df = pd.read_csv("twitter_sentiment_data.csv")
df = df[(df["sentiment"] != 0) & (df["sentiment"] != 2)]
texts = df["message"].astype(str).tolist()
y = df["sentiment"]
print("Twitter Climate Change Sentiment Dataset (after removing News & Neutral data):", df.shape)
print("Count of Believer Tweets:", (df["sentiment"] == 1).sum())
print("Count of Denier Tweets:", (df["sentiment"] == -1).sum())

# NOTE: Used ChatGPT/Google Gemini to assist with writing code for this step
# APPROACH 1 - SentenceBERT: SBERT generates embeddings using Siamese networks
# to compare pairs of sentences
sbert = SentenceTransformer("all-MiniLM-L6-v2")

# APPROACH 2 - SentenceBERT + contrastive pairs: fine-tune SBERT on sarcastic vs.
# non-sarcastic tweet pairs (iSarcasm or SARC dataset) before using it for stance

# NOTE: Used ChatGPT/Google Gemini to assist with writing code for this step
def build_isarcasm_pairs(df, num_samples = 8000):
    isarcasm_pairs = []
    df = df.dropna(subset = ["tweet", "rephrase"]).reset_index(drop = True)
    for i in range(min(num_samples, len(df))):
        tweet = str(df.iloc[i]["tweet"])
        rephrase = str(df.iloc[i]["rephrase"])
        # same meaning across sarcastic tweet and literal rephrase
        isarcasm_pairs.append(InputExample(texts = [tweet, rephrase]))
    return isarcasm_pairs

# NOTE: Used ChatGPT/Google Gemini to assist with writing code for this step
isarcasm_df = pd.read_csv("train.En.csv")
print("iSarcasm Dataset:", isarcasm_df.shape)
print("Count of Sarcastic Tweets:", (isarcasm_df["sarcastic"] == 1).sum())
print("Count of Non-Sarcastic Tweets:", (isarcasm_df["sarcastic"] == 0).sum())
train_examples = build_isarcasm_pairs(isarcasm_df, num_samples = 8000)
train_data_loader = DataLoader(train_examples, shuffle = True, batch_size = 32)

# NOTE: Used ChatGPT/Google Gemini to assist with writing code for this step
sbert_fine_tuned = SentenceTransformer("all-MiniLM-L6-v2")
train_loss = losses.MultipleNegativesRankingLoss(sbert_fine_tuned)
sbert_fine_tuned.fit(train_objectives = [(train_data_loader, train_loss)],
                     epochs = 5, show_progress_bar = True,
                     warmup_steps = int(len(train_data_loader) * 0.1))

# NOTE: Used ChatGPT/Google Gemini to assist with writing code for this step
# classification of both approaches using logistic regression (stable + strong)
def evaluate_approach(texts, y, model, name):
    cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)
    y_pred = np.zeros(len(y), dtype = int)
    for train_index, test_index in cv.split(texts, y):
        X_train = model.encode([texts[i] for i in train_index], batch_size = 32, show_progress_bar = True)
        X_test = model.encode([texts[i] for i in test_index], batch_size = 32, show_progress_bar = True)
        X_train = normalize(X_train)
        X_test = normalize(X_test)
        classifier = LogisticRegression(max_iter = 2000, class_weight = "balanced")
        classifier.fit(X_train, y.iloc[train_index])
        y_pred[test_index] = classifier.predict(X_test)
    print(f"\n=== {name} ===")
    print("Macro-F1:", np.round(f1_score(y, y_pred, average = "macro"), 4))
    print("Weighted-F1:", np.round(f1_score(y, y_pred, average = "weighted"), 4))
    print("Macro-Precision:", np.round(precision_score(y, y_pred, average = "macro"), 4))
    print("Macro-Recall:", np.round(recall_score(y, y_pred, average = "macro"), 4))
    print("Accuracy:", np.round(accuracy_score(y, y_pred), 4))
    print("Metrics by class:\n", classification_report(y, y_pred))
    return y_pred

# NOTE: Used ChatGPT/Google Gemini to assist with writing code for this step
y_pred_sbert = evaluate_approach(texts, y, sbert, "SentenceBERT (SBERT)")
y_pred_sbert_fine_tuned = evaluate_approach(texts, y, sbert_fine_tuned,
                  "SentenceBERT (SBERT) + Contrastive Pairs [iSarcasm Dataset Fine-Tuning]")

# NOTE: Used ChatGPT/Google Gemini to assist with writing code for this step
print(f"\n=== Classification Counts ===")
sbert_errors = np.sum(y_pred_sbert != y)
print(f"Total errors from SBERT: {sbert_errors}")
sbert_fine_tuned_errors = np.sum(y_pred_sbert_fine_tuned != y)
print(f"Total errors from SBERT with contrastive pairs: {sbert_fine_tuned_errors}")
better_by_fine_tuning = np.sum((y_pred_sbert != y) & (y_pred_sbert_fine_tuned == y))
print(f"Errors fixed by SBERT with contrastive pairs (IMPROVED): {better_by_fine_tuning}")
worse_by_fine_tuning = np.sum((y_pred_sbert == y) & (y_pred_sbert_fine_tuned != y))
print(f"Errors introduced by SBERT with contrastive pairs (DEGRADED): {worse_by_fine_tuning}")
same_by_fine_tuning = np.sum((y_pred_sbert != y) & (y_pred_sbert_fine_tuned != y))
print(f"Unchanged errors across SBERT and SBERT with contrastive pairs (CONSTANT): {same_by_fine_tuning}")

# NOTE: Used ChatGPT/Google Gemini to assist with writing code for this step
print(f"\n=== Error Analysis: SBERT vs. SBERT with Contrastive Pairs ===")
improvement = better_by_fine_tuning / sbert_errors * 100
print(f"SBERT Improvement Rate from Contrastive Pairs = {improvement:.2f}%")
degradation = worse_by_fine_tuning / len(y) * 100
print(f"SBERT Degradation Rate from Contrastive Pairs = {degradation:.2f}%")
net_error_change = (sbert_fine_tuned_errors - sbert_errors) / len(y) * 100
print(f"SBERT Net Error Change from Contrastive Pairs = {net_error_change:.2f}%")

# NOTE: Used ChatGPT/Google Gemini to assist with writing code for this step
print(f"\n=== SBERT Misclassified Tweets ===")
error_index = np.where(((y == 1) & (y_pred_sbert == -1)) | ((y == -1) & (y_pred_sbert == 1)))[0]
for i in error_index[:20]:
    print("-" * 100)
    print(f"Tweet: {texts[i]}")
    print(f"True Label: {y.iloc[i]} | Predicted Label: {y_pred_sbert[i]}")

# NOTE: Used ChatGPT/Google Gemini to assist with writing code for this step
print(f"\n=== SBERT Misclassified Tweets Improved by SBERT + Contrastive Pairs ===")
denier_errors = np.sum((y == -1) & (y_pred_sbert != y))
denier_improved = np.sum((y == -1) & (y_pred_sbert != y) & (y_pred_sbert_fine_tuned == y))
denier_perc = denier_improved / denier_errors * 100
print(f"\nPercentage of SBERT Denier Misclassifications Fixed by Contrastive Pairs: "
      f"{denier_perc:.2f}%")
believer_errors = np.sum((y == 1) & (y_pred_sbert != y))
believer_improved = np.sum((y == 1) & (y_pred_sbert != y) & (y_pred_sbert_fine_tuned == y))
believer_perc = believer_improved / believer_errors * 100
print(f"Percentage of SBERT Believer Misclassifications Fixed by Contrastive Pairs: "
      f"{believer_perc:.2f}%\n")
improved_index = np.where(((y == 1) & (y_pred_sbert != -1) & (y_pred_sbert_fine_tuned == 1)) |
                          ((y == -1) & (y_pred_sbert == 1) & (y_pred_sbert_fine_tuned == -1)))[0]
for i in improved_index[:20]:
    print("-" * 100)
    print(f"Tweet: {texts[i]}\n")
    print(f"True Label: {y.iloc[i]} | Predicted Label: {y_pred_sbert_fine_tuned[i]}")


Twitter Climate Change Sentiment Dataset (after removing News & Neutral data): (26952, 3)
Count of Believer Tweets: 22962
Count of Denier Tweets: 3990


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7129.40it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


iSarcasm Dataset: (3468, 10)
Count of Sarcastic Tweets: 867
Count of Non-Sarcastic Tweets: 2601


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14697.33it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss


Batches: 100%|██████████| 169/169 [00:04<00:00, 38.10it/s]



=== SentenceBERT (SBERT) ===
Macro-F1: 0.7133
Weighted-F1: 0.8274
Macro-Precision: 0.6892
Macro-Recall: 0.8068
Accuracy: 0.8049
Metrics by class:
               precision    recall  f1-score   support

          -1       0.42      0.81      0.55      3990
           1       0.96      0.80      0.88     22962

    accuracy                           0.80     26952
   macro avg       0.69      0.81      0.71     26952
weighted avg       0.88      0.80      0.83     26952



Batches: 100%|██████████| 169/169 [00:04<00:00, 39.26it/s]



=== SentenceBERT (SBERT) + Contrastive Pairs [iSarcasm Dataset Fine-Tuning] ===
Macro-F1: 0.7111
Weighted-F1: 0.826
Macro-Precision: 0.6874
Macro-Recall: 0.804
Accuracy: 0.8034
Metrics by class:
               precision    recall  f1-score   support

          -1       0.42      0.80      0.55      3990
           1       0.96      0.80      0.87     22962

    accuracy                           0.80     26952
   macro avg       0.69      0.80      0.71     26952
weighted avg       0.88      0.80      0.83     26952


=== Classification Counts ===
Total errors from SBERT: 5259
Total errors from SBERT with contrastive pairs: 5299
Errors fixed by SBERT with contrastive pairs (IMPROVED): 407
Errors introduced by SBERT with contrastive pairs (DEGRADED): 447
Unchanged errors across SBERT and SBERT with contrastive pairs (CONSTANT): 4852

=== Error Analysis: SBERT vs. SBERT with Contrastive Pairs ===
SBERT Improvement Rate from Contrastive Pairs = 7.74%
SBERT Degradation Rate from Contrasti